In [ ]:
import os
import json
import pandas as pd
import numpy as np

# Benchmark experiment

In [ ]:
files = {
    "warehouse1": [
        # "replan_warehouse-20-40-10-2-1-random-12-k200_paths_2026-03-17_seed123_66delays"
    ],
    "maze1": [
        "maze-128-128-1-even-1-k20_paths_2026-03-16_seed123_k20"
    ]
}
all_results = {}
for map_name in files:
    all_results[map_name] = {}
    for filename in files[map_name]:
        num_agents = filename.split("_")[-1]
        filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", map_name, f"{filename}.json")
        all_results[map_name][num_agents] = json.load(open(filepath))

In [ ]:
df = pd.DataFrame(columns=["map", "num_agents", "num_agents_tested", "avg_search_time_flexsipp", "avg_generation_time_flexsipp", "avg_found_paths_flexsipp", "avg_search_time_maeder", "avg_generation_time_maeder", "avg_found_paths_maeder", "avg_delay_improvement"])
num_rows = 0
for map_name in all_results:
    for num_agents in all_results[map_name]:
        maeder = {"search": [], "generation": [], "paths": [], "delays": {a: [] for a in all_results[map_name][num_agents]}}
        flexsipp = {"search": [], "generation": [], "paths": [], "delays": {a: [] for a in all_results[map_name][num_agents]}}
        for delay_agent in all_results[map_name][num_agents]:
            print(map_name, num_agents, delay_agent, all_results[map_name][num_agents][delay_agent])
            if "FlexSIPP" in all_results[map_name][num_agents][delay_agent]:
                flexsipp["search"].append(all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["Search Time"])
                flexsipp["generation"].append(all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["gen_time"])
                flexsipp["paths"].append(len(all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["unique_routes_safe"]))
                for path, atf_strings in all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["unique_routes_safe"].items():
                    for atf_str in atf_strings:
                        atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                        current_delay = (atf[1] + atf[3]) - all_results[map_name][num_agents][delay_agent]["original_arrival_time"]
                        for t, t_atf, t_path, other_delays in all_results[map_name][num_agents][delay_agent]["FlexSIPP"]["tipping_points"]:
                            if t_path == path:
                                current_delay += sum([min([d for x, d in other_delays[a].items()]) if other_delays[a] else 0 for a in other_delays])
                        flexsipp["delays"][delay_agent].append(current_delay)
                maeder["search"].append(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["Search Time"])
                maeder["generation"].append(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["gen_time"])
                maeder["paths"].append(len(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["unique_routes_safe"]))
                for path, atf_strings in all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["unique_routes_safe"].items():
                    for atf_str in atf_strings:
                        atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                        current_delay = (atf[1] + atf[3]) - all_results[map_name][num_agents][delay_agent]["original_arrival_time"]
                        maeder["delays"][delay_agent].append(current_delay)
        df.loc[num_rows] = [
            map_name,
            num_agents, 
            len(maeder["search"]), 
            sum(flexsipp["search"]) / len(flexsipp["search"]), 
            sum(flexsipp["generation"]) / len(flexsipp["generation"]), 
            sum(flexsipp["paths"]) / len(flexsipp["paths"]), 
            sum(maeder["search"]) / len(maeder["search"]), 
            sum(maeder["generation"]) / len(maeder["generation"]), 
            sum(maeder["paths"]) / len(maeder["paths"]),
            sum([min(maeder["delays"][a] if maeder["delays"][a] else [0]) - min(flexsipp["delays"][a]  if flexsipp["delays"][a] else [0]) for a in flexsipp["delays"]]) / len(flexsipp["delays"])
        ]
        num_rows += 1
df

# Replanning experiment


In [ ]:
replan_files = {
    "warehouse1": [
        "replan_FlexSIPP_warehouse1_2026-03-18_seed123",
        "replan_@MAEDeR_warehouse1_2026-03-18_seed123"
    ],
    "maze1": [
        "replan_FlexSIPP_maze1_2026-03-18_seed123",
        "replan_@MAEDeR_maze1_2026-03-18_seed123"
    ]
}
all_results = {}
for map_name in replan_files:
    all_results[map_name] = {}
    for file in replan_files[map_name]:
        algorithm = file.split("_")[1]
        filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", map_name, f"{file}.json")
        obj = json.load(open(filepath))
        for scen, data in obj.items():
            if scen in all_results[map_name]:
                all_results[map_name][scen][algorithm] = data
            else:
                all_results[map_name][scen]= {algorithm: data}

In [ ]:
df = pd.DataFrame(columns=["map", "scenario", "num_delays", "num_agents_tested_flexsipp", "num_agents_test_maeder", "avg_search_time_flexsipp", "avg_preprocess_time_flexsipp", "avg_postprocess_time_flexsipp", "avg_found_paths_flexsipp", "avg_search_time_maeder", "avg_preprocess_time_maeder", "avg_postprocess_time_maeder", "avg_found_paths_maeder", "avg_delay_improvement"])
delay_improvement = pd.DataFrame(columns=["map", "scenario", "delay", "delay_agent", "original_arrival_time_flexsipp", "original_arrival_time_maeder", "delay_flexsipp", "delay_maeder", "delta_flexsipp", "delta_maeder"])
num_rows = 0
delay_rows = 0
for map_name in all_results:
    for scen_file in all_results[map_name]:
        maeder = {"search": [], "preprocess": [], "postprocess": [], "paths": [], "not_found": 0, "delays": {a: [] for a in all_results[map_name][scen_file]["@MAEDeR"]}}
        flexsipp = {"search": [], "preprocess": [], "postprocess": [], "paths": [], "not_found":0, "delays": {a: [] for a in all_results[map_name][scen_file]["FlexSIPP"]}}
        num_delays = len(all_results[map_name][scen_file]["FlexSIPP"])
        if "FlexSIPP" in all_results[map_name][scen_file]:
            for delay_agent in all_results[map_name][scen_file]["FlexSIPP"]:
                flex_delay = []
                if not all_results[map_name][scen_file]["FlexSIPP"][delay_agent]:
                    print("No results for", map_name, scen_file, delay_agent, "flexsipp")
                    flexsipp["not_found"] += 1
                else:
                    flexsipp["search"].append(all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["Search Time"])
                    flexsipp["preprocess"].append(all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["preprocess_time"])
                    flexsipp["postprocess"].append(all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["postprocess_time"])
                    flexsipp["paths"].append(len(all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["unique_routes_safe"]))
                    for path, atf_strings in all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["unique_routes_safe"].items():
                        for atf_str in atf_strings:
                            atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                            current_delay = (atf[1] + atf[3]) - all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["original_arrival_time"]
                            flexsipp["delays"][delay_agent].append(current_delay)
                            flex_delay = [all_results[map_name][scen_file]["FlexSIPP"][delay_agent]["original_arrival_time"], current_delay, atf[3]]
                if not all_results[map_name][scen_file]["@MAEDeR"][delay_agent]:
                    print("No results for", map_name, scen_file, delay_agent, "maeder")
                    maeder["not_found"] += 1 
                else:
                    maeder["search"].append(all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["Search Time"])
                    maeder["preprocess"].append(all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["preprocess_time"])
                    maeder["postprocess"].append(all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["postprocess_time"])
                    maeder["paths"].append(len(all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["unique_routes_safe"]))
                    for path, atf_strings in all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["unique_routes_safe"].items():
                        for atf_str in atf_strings:
                            atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                            current_delay = (atf[1] + atf[3]) - all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["original_arrival_time"]
                            maeder["delays"][delay_agent].append(current_delay)
                            if flex_delay:
                                delay_improvement.loc[delay_rows] = [
                                    map_name,
                                    scen_file,
                                    delay_agent,
                                    all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["delay_agent"],
                                    flex_delay[0],
                                    all_results[map_name][scen_file]["@MAEDeR"][delay_agent]["original_arrival_time"],
                                    flex_delay[1],
                                    current_delay,
                                    flex_delay[2],
                                    atf[3]
                                ]
                                delay_rows += 1  
            # df.loc[num_rows] = [
            #     map_name, 
            #     scen_file,
            #     num_delays,
            #     num_delays - flexsipp["not_found"],
            #     num_delays - maeder["not_found"],
            #     sum(flexsipp["search"]) / len(flexsipp["search"]), 
            #     sum(flexsipp["preprocess"]) / len(flexsipp["preprocess"]),
            #     sum(flexsipp["postprocess"]) / len(flexsipp["postprocess"]),
            #     sum(flexsipp["paths"]) / len(flexsipp["paths"]), 
            #     sum(maeder["search"]) / len(maeder["search"]), 
            #     sum(maeder["preprocess"]) / len(maeder["preprocess"]),
            #     sum(maeder["postprocess"]) / len(maeder["postprocess"]),
            #     sum(maeder["paths"]) / len(maeder["paths"]),
            #     sum([min(maeder["delays"][a] if maeder["delays"][a] else [0]) - min(flexsipp["delays"][a]  if flexsipp["delays"][a] else [0]) for a in flexsipp["delays"]]) / len(flexsipp["delays"])
            # ]
            # num_rows += 1
df

In [ ]:
delay_improvement